# VMBench workshop

Classify contest videos as **real (0)** or **generated (1)**, then explore motion diagnostics.

Components: **PAS** amplitude · **OIS** human structure · **TCS** coherence · **CAS** learned plausibility.

In [ ]:
from pathlib import Path
if not Path("/content/workshop/.git").exists():
    !git clone --recursive https://github.com/KK-one/vmbench-workshop.git /content/workshop
%cd /content/workshop

## 0. Initialization

Configure optional metrics and video selection. Set `RUN_METRICS=False` to only explore videos and prepare submissions.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "utils.py").is_file():
    PROJECT_DIR = Path("/content/workshop")
sys.path.insert(0, str(PROJECT_DIR))
from utils import (WorkshopConfig, configure, initialize, download_contest, contest_video_paths,
                   browse_contest, write_binary_submission, compute_pas, compute_ois,
                   compute_tcs, compute_cas, compare_variants)

WORK_DIR = Path("/content")
COMPETITION = ""  # Paste the competition slug or /competitions/<slug> URL; blank prompts.
RUN_METRICS = True
METRICS_TO_RUN = ["PAS", "OIS", "TCS", "CAS"]
METRIC_VIDEO_IDS = []  # Choose HEX IDs after browsing; blank selects the first two.
MAX_METRIC_VIDEOS = 2
VIDEO_META = {}  # filename -> {"subject_noun": "person", "subject_type": "human", "prompt": "..."}
CREATE_VARIANTS = False
workshop = configure(WorkshopConfig(work_dir=WORK_DIR, repo_dir=PROJECT_DIR / "vendor/vmbench",
                                   metrics_to_run=METRICS_TO_RUN, video_meta=VIDEO_META))

## 1. Download contest videos

[Join the competition](https://www.kaggle.com/t/ac5704b939234a8d96a93d8aabc66bde) and accept its rules first. Get a personal token from [Kaggle API settings](https://www.kaggle.com/settings/api). Save it as the Colab secret `KAGGLE_API_TOKEN`, or enter it in the masked prompt.

Downloads are cached per runtime. Nested ZIPs are unpacked automatically; original video bytes are preserved.

In [ ]:
CONTEST_DIR = download_contest(COMPETITION, WORK_DIR / "kaggle_data")
CONTEST_VIDEO_PATHS = contest_video_paths(CONTEST_DIR)
print(f"Contest: {len(CONTEST_VIDEO_PATHS)} videos")

## 2. Browse and create a baseline

Inspect videos and record your guesses before computing metrics. The baseline predicts **real (0)** for every video; it covers the complete contest, regardless of the metric subset.

In [ ]:
video_browser = browse_contest(CONTEST_VIDEO_PATHS)
baseline_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "baseline_all_real.csv")
from IPython.display import FileLink, display
display(FileLink(str(baseline_path)))
# Download when ready:
# from google.colab import files
# files.download(str(baseline_path))

## 3. Set up selected metrics

Select T4 GPU for live metrics. Only the chosen evaluators and checkpoints are installed. Set `VIDEO_META` to the visible subject for PAS/TCS; OIS applies to people. These scores are diagnostics, not real/generated probabilities. MSS is excluded from this T4 workshop.

In [ ]:
VIDEO_PATH, VIDEO_PATHS = None, []
if RUN_METRICS:
    if not isinstance(MAX_METRIC_VIDEOS, int) or MAX_METRIC_VIDEOS < 1:
        raise ValueError("MAX_METRIC_VIDEOS must be a positive integer.")
    if not METRICS_TO_RUN or set(METRICS_TO_RUN) - {"PAS", "OIS", "TCS", "CAS"}:
        raise ValueError("Select PAS, OIS, TCS and/or CAS for this T4 notebook.")
    available = {p.stem: p for p in CONTEST_VIDEO_PATHS}
    selected_ids = METRIC_VIDEO_IDS or list(available)[:MAX_METRIC_VIDEOS]
    if len(set(selected_ids)) != len(selected_ids) or set(selected_ids) - set(available):
        raise ValueError("Select unique video IDs from the downloaded contest.")
    workshop = initialize(PROJECT_DIR, metrics=METRICS_TO_RUN, video_meta=VIDEO_META,
                          create_variants=CREATE_VARIANTS, work_dir=WORK_DIR)
    workshop.input_paths = [available[video_id] for video_id in selected_ids]
    video_info_df = workshop.preview_videos()
    VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()
else:
    print("Live metrics skipped. Videos and the all-real baseline are ready.")

## 4. Speed experiment (optional)

Enable `CREATE_VARIANTS` for fast ×2, slow ×0.5, frame drop and duplication. Variants are experiments, not extra contest entries.

In [ ]:
if RUN_METRICS and CREATE_VARIANTS:
    VARIANTS = workshop.prepare_variants()
    VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()

## PAS

Subject displacement with camera-motion compensation. More motion does not establish authenticity.

In [ ]:
if RUN_METRICS and "PAS" in METRICS_TO_RUN:
    pas = compute_pas(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("PAS")

## OIS

Human structural integrity. N/A when a person or pose cannot be detected.

In [ ]:
if RUN_METRICS and "OIS" in METRICS_TO_RUN:
    ois = compute_ois(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("OIS")

## TCS

Object persistence. Check detection warnings when interpreting high scores.

In [ ]:
if RUN_METRICS and "TCS" in METRICS_TO_RUN:
    tcs = compute_tcs(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("TCS")

## CAS

Learned plausibility, not a physics simulator or a calibrated real/generated classifier.

In [ ]:
if RUN_METRICS and "CAS" in METRICS_TO_RUN:
    cas = compute_cas(VIDEO_PATH)
    if CREATE_VARIANTS:
        compare_variants("CAS")

## Results

Evaluate only the selected subset. Missing scores are not zeros; inspect status and runtime.

In [ ]:
if RUN_METRICS:
    workshop.show_metric_summary(VIDEO_PATH)
    results_df = workshop.evaluate_dataset()
    workshop.plot_results(results_df)

## Submit to Kaggle

Download `baseline_all_real.csv` and edit its `label` column, or fill `PREDICTIONS` below for every contest ID. Upload the CSV to Kaggle manually. Accuracy = correct labels / all videos; a perfect submission scores 1.0.

In [ ]:
PREDICTIONS = {}  # Example format: {"a1b2c3": 0, "d4e5f6": 1}; use actual contest IDs.
if PREDICTIONS:
    submission_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "submission.csv",
                                              predictions=PREDICTIONS)
    display(FileLink(str(submission_path)))
else:
    print("Edit the baseline CSV or fill PREDICTIONS for every contest video.")

## Discuss

- Did any metric change your initial guess?
- Where did a score disagree with your visual judgment?
- Why can a generated video receive high motion scores?